In [ ]:
import os, sys, json, warnings
import time
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from xgboost import XGBRegressor

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_preprocessing as du
import utils.model_utils as mu

importlib.reload(du)
importlib.reload(mu)

In [ ]:
# ============================================================
# Configuration — everything set manually lives here (D012).
#
# The search runs once per stock x price-measure x horizon; each
# trial trains on day d and is scored on the FULL day d+1, averaged
# over the day pairs. All winning params (incl. tree count) are
# frozen; the training pipeline starts strictly after TUNE_DATES.
# Cost = N_TRIALS x N_PAIRS x n_targets fits per stock.
# ============================================================

# Stocks to tune (every stock gets its own search).
TUNE_SYMBOLS = du.SYMBOLS[:1]

# Early time block used for tuning. Excluded from headline results;
# recorded in the output json so xgboost.ipynb starts after it.
TUNE_DATES = du.SAMPLE_DATES[:10]

N_TRIALS = 40 # Random-search trials per stock-target.

N_PAIRS = 5 # (train day, validation day) pairs per trial, spread evenly over TUNE_DATES.

SEED = 0

HORIZONS = ["100ms", "2s", "30s", "5m"] # Return horizons defining the targets.

# Searched hyperparameters: name -> (distribution, low, high).
# "log"/"logint" sample log-uniformly (right choice for scale-like
# params); "int"/"logint" round to integers.
SEARCH_SPACE = {
    "max_depth": ("int", 3, 8),
    "learning_rate": ("log", 0.01, 0.3),
    "min_child_weight": ("logint", 5, 100),
    "subsample": ("uniform", 0.5, 1.0),
    "colsample_bytree": ("uniform", 0.5, 1.0),
    "reg_lambda": ("log", 0.01, 10.0),
}

# Fixed (non-searched) model settings for every trial fit.
# n_estimators is only an upper bound: early stopping on the
# validation day picks the actual tree count, which gets frozen.
BASE_PARAMS = dict(
    n_estimators=2000,
    early_stopping_rounds=50,
    eval_metric="rmse",
    tree_method="hist",
    max_bin=128,
    device=mu.select_device(),   # least-used GPU, else "cpu"
    n_jobs=-1,
    random_state=0,
)
print(f"XGBoost device: {BASE_PARAMS['device']}")

In [ ]:
# ============================================================
# XGBoost-specific search functions
# ============================================================

def sample_params(rng: np.random.Generator) -> dict:
    """Draw one random-search trial from SEARCH_SPACE."""
    params = {}
    for name, (kind, lo, hi) in SEARCH_SPACE.items():
        if kind == "int":
            params[name] = int(rng.integers(lo, hi + 1))
        elif kind == "uniform":
            params[name] = float(rng.uniform(lo, hi))
        elif kind == "log":
            params[name] = float(np.exp(rng.uniform(np.log(lo), np.log(hi))))
        elif kind == "logint":
            params[name] = int(round(np.exp(rng.uniform(np.log(lo), np.log(hi)))))
        else:
            raise ValueError(f"Unknown distribution kind: {kind}")
    return params


def random_search(
        pairs: list,
        base_params: dict,
        n_trials: int,
        seed: int = 0,
) -> pd.DataFrame:
    """
    Seeded random search for one target (1D y) with day-pair validation.

    pairs: list of (X_tr, y_tr, X_val, y_val) tuples: train on day d,
    validate on the FULL following day d+1. This mirrors the walk-forward
    deployment (same training-set size, same one-day-ahead task) and avoids
    the pre-close bias a same-day tail slice would have (intraday volatility
    is U-shaped, so end-of-day rows are not representative).

    Each trial fits one model per pair with early stopping on the validation
    day (base_params should set a large n_estimators and
    early_stopping_rounds); the trial's score is the mean validation RMSE
    across pairs, and `n_estimators_frozen` is the median early-stopped
    best_iteration: the tree count to freeze alongside the winning params.
    """
    rng = np.random.default_rng(seed)
    rows = []

    for trial in range(n_trials):
        params = sample_params(rng)
        scores, iterations = [], []

        start = time.perf_counter()
        for X_tr, y_tr, X_val, y_val in pairs:
            model = XGBRegressor(**{**base_params, **params})
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            scores.append(float(model.best_score))
            iterations.append(int(model.best_iteration))

        rows.append({
            "trial": trial,
            **params,
            "mean_val_rmse": float(np.mean(scores)),
            "n_estimators_frozen": int(np.median(iterations)),
            "fit_seconds": time.perf_counter() - start,
        })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# Derived setup: nothing to configure below this line
# ============================================================
warnings.filterwarnings("ignore")

PARENT = os.path.dirname(os.getcwd())
DATA_ROOT = f"{PARENT}/data/processed"
TUNING_DIR = f"{PARENT}/model_outputs/XGBoost/Tuning"

sample_cols = pd.read_parquet(
    f"{DATA_ROOT}/{TUNE_SYMBOLS[0]}/{TUNE_DATES[0]}.parquet"
).columns
FEATURE_COLS = list(sample_cols[sample_cols.str.startswith("F_")])
TARGET_COLS = [c for c in sample_cols
               if c.startswith("T_") and c.rsplit("_", 1)[-1] in HORIZONS]

# Evenly spread pair start indices over the block (pair i = days i, i+1)
PAIR_IDX = sorted(set(
    np.linspace(0, len(TUNE_DATES) - 2, N_PAIRS).round().astype(int)
))
print("day pairs:", [(TUNE_DATES[i], TUNE_DATES[i+1]) for i in PAIR_IDX])

In [ ]:
# ============================================================
# Per-stock, per-target random search — checkpointed per stock:
# each stock's trials are written to TRIALS_DIR/<symbol>.parquet as
# soon as the stock finishes, and stocks with an existing file are
# skipped. To resume after a crash, just re-run this cell; to redo a
# stock from scratch, delete its parquet first.
# ============================================================
TRIALS_DIR = f"{TUNING_DIR}/trials_seed{SEED}"

start = time.perf_counter()

for symbol in tqdm(TUNE_SYMBOLS, desc="Stocks", unit="stock"):
    if os.path.exists(f"{TRIALS_DIR}/{symbol}.parquet"):
        tqdm.write(f"{symbol}: checkpoint found, skipping")
        continue

    # Each stock is tuned on its own early block (no pooling).
    cache = mu.load_day_cache(DATA_ROOT, symbol, TUNE_DATES, FEATURE_COLS, TARGET_COLS)
    symbol_trials = []

    for j, target in enumerate(tqdm(TARGET_COLS, desc=symbol, leave=False, unit="target")):
        pairs = [
            (cache[TUNE_DATES[i]]["X"], cache[TUNE_DATES[i]]["Y"][:, j],
             cache[TUNE_DATES[i+1]]["X"], cache[TUNE_DATES[i+1]]["Y"][:, j])
            for i in PAIR_IDX
        ]

        trials = random_search(
            pairs,
            base_params=BASE_PARAMS,
            n_trials=N_TRIALS,
            seed=SEED,
        )
        trials.insert(0, "target", target)
        trials.insert(0, "symbol", symbol)
        symbol_trials.append(trials)

    del cache

    mu.save_table(
        df=pd.concat(symbol_trials, ignore_index=True),
        root_dir=TRIALS_DIR,
        filename=f"{symbol}.parquet",
    )

tqdm.write(f"TOTAL SEARCH TIME: {time.perf_counter()-start:.2f}s")

# Collect all checkpoints (incl. earlier runs') for the freeze step.
all_trials = pd.concat(
    [pd.read_parquet(f"{TRIALS_DIR}/{symbol}.parquet") for symbol in TUNE_SYMBOLS],
    ignore_index=True,
)

In [ ]:
# ============================================================
# Freeze the winners -> best_params json: {"tune_dates": [...], "params": {symbol: {target: params}}}
# tune_dates records the tuning block so the training pipeline can start strictly after it.
# ============================================================
PARAM_NAMES = list(SEARCH_SPACE)

best_rows = all_trials.loc[
    all_trials.groupby(["symbol", "target"])["mean_val_rmse"].idxmin()
]

best_params = {}
for _, row in best_rows.iterrows():
    params = {
        name: (int(row[name]) if SEARCH_SPACE[name][0] in ("int", "logint") else float(row[name]))
        for name in PARAM_NAMES
    }
    params["n_estimators"] = max(1, int(row["n_estimators_frozen"]))
    best_params.setdefault(row["symbol"], {})[row["target"]] = params

os.makedirs(TUNING_DIR, exist_ok=True)
with open(f"{TUNING_DIR}/best_params_seed{SEED}.json", "w") as f:
    json.dump({"tune_dates": list(TUNE_DATES), "params": best_params}, f, indent=2)

best_rows[["symbol", "target", "mean_val_rmse", "n_estimators_frozen"] + PARAM_NAMES]